<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Composing agents with NVIDIA NeMo Fabric

This is the **advanced** notebook. The [quickstart](01_quickstart.ipynb) is
fully self-contained — it builds every agent inline. This notebook instead
**builds on the maintained [code-review example](../code_review_agent/README.md)**,
using its `base_config()` as the portable agent it composes on. That is the
intended split: learn the mechanics standalone, then see composition against
a real, maintained agent.

Composition falls into two groups:

1. **Harness variation** — run the *same* logical agent on a different
   harness (Hermes Agent, Codex, Claude, Deep Agents). NeMo Fabric's primary purpose.
2. **Agent configuration variation** — vary fields of the `FabricConfig`
   itself: skills, MCP servers, models, telemetry, and so on.

Terms used consistently below: a **harness** is the agent framework (Hermes Agent,
Codex, Claude, Deep Agents); an **adapter** is the NeMo Fabric integration that
connects NeMo Fabric to a harness; the **runtime** is NeMo Fabric's own start/invoke/
stop lifecycle, which does not change when you switch harnesses.

Where a harness's visible prerequisites are present, the cells below run it;
otherwise they inspect the resolved config with `plan()` and say what to
provide. Attempted-run failures are collected so every variant is tried, then
raised after the remaining configuration examples execute.

<div align="center">
<img src="img/variations.svg" width="620" alt="Two ways to vary an agent with NeMo Fabric: harness variation and agent configuration variation.">
<br><br>
<em>Two kinds of variation on one typed agent — run it on a different harness, or vary <code>FabricConfig</code> fields like skills, MCP servers, models, and telemetry.</em>
</div>

## Setup

Load the environment and the maintained example's `base_config`. Each
harness adapter runs in whatever Python environment has that adapter
installed; the run loop selects that interpreter through `ADAPTER_PYTHON`.
The next cell uses a local NeMo Fabric build when present and otherwise
installs the published package and package-installable harnesses used in this
notebook. Hermes Agent 0.20 and later is no longer installable from PyPI, so
the setup cell checks out the v0.20.1 commit and installs it as an editable
package with the selected Hermes adapter interpreter, defaulting to the active
notebook kernel. In
Colab, the setup also clones the repository because the maintained example
is not part of the Python package. It also provisions Codex authentication
from an OpenAI Platform API key. The setup then prompts for any missing API
keys; press Enter to skip a key and the harnesses that require it.

In [ ]:
import os
import sys

os.environ["FABRIC_PYTHON"] = sys.executable
os.environ["HERMES_PYTHON"] = os.environ.get("ADAPTER_PYTHON", sys.executable)

In [ ]:
%%bash
set -euo pipefail

NEMO_FABRIC_VERSION="0.3.0"
"$FABRIC_PYTHON" -m pip install "nemo-fabric[claude,codex,deepagents,relay]==$NEMO_FABRIC_VERSION"

if [ "$("$HERMES_PYTHON" -c 'from importlib.metadata import version; print(version("nemo-fabric-adapters-hermes"))' 2>/dev/null || true)" != "$NEMO_FABRIC_VERSION" ]; then
    "$HERMES_PYTHON" -m pip install "nemo-fabric-adapters-hermes==$NEMO_FABRIC_VERSION"
fi

# f80f453ae0679347e38abc917c7f94f717bf96c5 aligns with Hermes Agent v0.20.1.
HERMES_COMMIT="f80f453ae0679347e38abc917c7f94f717bf96c5"
REPO_ROOT="$(git rev-parse --show-toplevel 2>/dev/null || pwd)"
HERMES_CHECKOUT="$REPO_ROOT/external/hermes-agent"

if [ -e "$HERMES_CHECKOUT" ] && [ ! -d "$HERMES_CHECKOUT/.git" ]; then
    echo "ERROR: expected a Git checkout at $HERMES_CHECKOUT" >&2
    exit 1
fi
if [ ! -d "$HERMES_CHECKOUT/.git" ]; then
    mkdir -p "$(dirname "$HERMES_CHECKOUT")"
    git init --quiet "$HERMES_CHECKOUT"
    git -C "$HERMES_CHECKOUT" remote add origin https://github.com/NousResearch/hermes-agent.git
elif ! git -C "$HERMES_CHECKOUT" diff --quiet || ! git -C "$HERMES_CHECKOUT" diff --cached --quiet; then
    echo "ERROR: Hermes Agent checkout has tracked changes: $HERMES_CHECKOUT" >&2
    exit 1
fi
git -C "$HERMES_CHECKOUT" fetch --depth 1 origin "$HERMES_COMMIT"
git -C "$HERMES_CHECKOUT" checkout --quiet --detach FETCH_HEAD
"$HERMES_PYTHON" -m pip install --config-settings editable_mode=compat --editable "$HERMES_CHECKOUT"

In [ ]:
import getpass
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


REPOSITORY_URL = "https://github.com/NVIDIA/NeMo-Fabric.git"
COLAB_REPO_ROOT = Path("/content/NeMo-Fabric")


def running_in_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    root = start
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent
    if (root / "pyproject.toml").exists():
        return root

    if not running_in_colab():
        return start

    if not (COLAB_REPO_ROOT / "pyproject.toml").exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                REPOSITORY_URL, str(COLAB_REPO_ROOT),
            ],
            check=True,
        )
    return COLAB_REPO_ROOT


def provision_codex_api_key(api_key: str) -> None:
    from openai_codex import Codex

    with Codex() as codex:
        codex.login_api_key(api_key)


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.removeprefix("export ").partition("=")
        key = key.strip()
        if key and key not in os.environ:
            os.environ[key] = value.strip().strip("'\"")


IN_COLAB = running_in_colab()
REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

api_key_names = ["NVIDIA_API_KEY", "ANTHROPIC_API_KEY"]
if IN_COLAB:
    print("Codex in Colab uses an OpenAI Platform API key.")
    api_key_names.append("OPENAI_API_KEY")
for api_key_name in api_key_names:
    if api_key_name not in os.environ:
        api_key = getpass.getpass(
            f"Enter your {api_key_name}, or press Enter to skip: "
        )
        if len(api_key) > 0:
            os.environ[api_key_name] = api_key

if IN_COLAB and os.environ.get("OPENAI_API_KEY"):
    provision_codex_api_key(os.environ["OPENAI_API_KEY"])

from nemo_fabric import (
    Fabric,
    HarnessConfig,
    InstructionConfig,
    InstructionsConfig,
    ModelConfig,
    RelayAtofConfig,
    RelayAtofFileSinkConfig,
    RelayObservabilityConfig,
)

# Build on the maintained code-review example: base_config() is the portable
# agent this notebook composes on, and BASE_DIR anchors its relative paths.
from examples.code_review_agent import base_config, BASE_DIR

# A harness adapter runs in whatever Python environment has that adapter
# installed. By default all adapters use this notebook's interpreter; set
# ADAPTER_PYTHON when Hermes Agent is managed in a separate environment.
FABRIC_PY = sys.executable
HERMES_PY = os.environ.get("ADAPTER_PYTHON") or FABRIC_PY

WORKSPACE = "./repos/my-service"  # relative to BASE_DIR
INSTRUCTION = "You are a concise code reviewer. Point out correctness bugs and risks."

fabric = Fabric()
RELAY_AVAILABLE = importlib.util.find_spec("nemo_relay") is not None
print("example base_dir   :", BASE_DIR)
print("fabric interpreter :", FABRIC_PY)
print("hermes interpreter :", HERMES_PY or "not found")
print("nemo_relay present :", RELAY_AVAILABLE)


## Group 1 — Harness variation

Start from the example's `base_config()` and run the *same* agent on four
harnesses. To make this a **controlled** comparison, the portable base is
held constant — same identity, workspace, artifact root, and output contract
— and only the harness changes.

A few differences are unavoidable and are called out explicitly:

- **Model** — each harness talks to its own provider, so the model differs
  (Hermes Agent and Deep Agents use an NVIDIA-hosted model; Codex uses OpenAI;
  Claude uses Anthropic).
- **Input schema** — the Codex adapter takes `text`; the others take `chat`.
- **Instruction delivery** — all four harnesses receive normalized system
  instructions through `instructions.system`; Codex maps them to base instructions
  while its per-turn prompt remains `text` input.
- **Capabilities** — `base_config` ships a code-review skill, but not every
  harness accepts the same capabilities, so `for_harness` drops it here. That
  difference across harnesses is exactly why the
  example provides per-harness builders; Group 2 composes capabilities back.

In [ ]:
def for_harness(harness):
    """Adapt the maintained code-review agent to one harness.

    The portable base — identity, workspace, artifact root, output contract —
    comes from `base_config()` and is held constant. Only the harness and the
    values it forces to differ are changed here.
    """
    cfg = base_config()
    # base_config ships a code-review skill. Not every harness accepts every
    # capability (which is exactly why the example provides per-harness
    # builders), so drop it for an apples-to-apples comparison.
    # Group 2 composes capabilities back on.
    cfg.remove_skill_path("./skills/code-review")
    cfg.harness = HarnessConfig(
        adapter_id=harness["adapter_id"], resolution="preinstalled",
        settings=harness["settings"],
    )
    cfg.instructions = InstructionsConfig(
        system=InstructionConfig(content=INSTRUCTION, mode="replace"),
    )
    cfg.models = {"default": harness["model"]}
    cfg.runtime.max_turns = harness.get("max_turns")
    cfg.runtime.input_schema = harness["input_schema"]  # Codex requires "text"
    return cfg


NVIDIA_MODEL = ModelConfig(
    provider="nvidia", model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning",
    base_url="https://integrate.api.nvidia.com/v1",
    temperature=0.0, api_key_env="NVIDIA_API_KEY",
)

HARNESSES = [
    {"name": "Hermes", "adapter_id": "nvidia.fabric.hermes", "python": HERMES_PY,
     "input_schema": "chat", "model": NVIDIA_MODEL, "key": "NVIDIA_API_KEY",
     "needs": "the editable Hermes Agent checkout from the setup cell + NVIDIA_API_KEY",
     "max_turns": 1,
     "settings": {"max_tokens": 512,
                  "reasoning_config": {"effort": "none"}}},
    {"name": "Deep Agents", "adapter_id": "nvidia.fabric.langchain.deepagents", "python": FABRIC_PY,
     "input_schema": "chat", "model": NVIDIA_MODEL, "key": "NVIDIA_API_KEY",
     "needs": "the Deep Agents adapter (nemo-fabric[deepagents]) + NVIDIA_API_KEY",
     "settings": {}},
    {"name": "Codex", "adapter_id": "nvidia.fabric.codex", "python": FABRIC_PY,
     "input_schema": "text",
     **({"key": "OPENAI_API_KEY"} if IN_COLAB else {}),
     "model": ModelConfig(provider="openai", model="openai/gpt-5.4"),
     "needs": "the Codex adapter (nemo-fabric[codex]) + Codex authentication (validated when the adapter starts)",
     "settings": {"sandbox": "workspace-write", "reasoning_effort": "high"}},
    {"name": "Claude", "adapter_id": "nvidia.fabric.claude", "python": FABRIC_PY,
     "input_schema": "chat", "key": "ANTHROPIC_API_KEY",
     "model": ModelConfig(provider="anthropic", model="anthropic/claude-sonnet-4-5",
                          api_key_env="ANTHROPIC_API_KEY"),
     "needs": "the Claude adapter + ANTHROPIC_API_KEY",
     "max_turns": 1,
     "settings": {"permission_mode": "dontAsk"}},
]


def adapter_python_available(value):
    """Return whether an adapter interpreter path or command is available."""
    path = Path(value)
    if len(path.parts) == 1:
        return shutil.which(str(path)) is not None
    if not path.is_absolute():
        path = BASE_DIR / path
    return path.is_file() and os.access(path, os.X_OK)


def blocker(harness):
    """Return why a harness cannot run here, or None if it can."""
    py = harness.get("python")
    if not py or not adapter_python_available(py):
        return "adapter interpreter not found"
    # Codex authentication can use multiple credential stores, so its adapter
    # validates credentials when the run starts rather than in this helper.
    if harness.get("key") and not os.environ.get(harness["key"]):
        return f"{harness['key']} is not set"
    if harness.get("binary") and not shutil.which(harness["binary"]):
        return f"`{harness['binary']}` not on PATH"
    return None


def oneline(text, limit=200):
    s = " ".join(str(text).split())
    return s if len(s) <= limit else s[:limit] + " ..."


def failure_detail(result):
    return result.error.to_mapping() if result.error else {"status": result.status}


In [ ]:
PROMPT = "In one sentence, what does the Python expression sum(v) / len(v) compute, and name one risk?"

run_failures = []
for harness in HARNESSES:
    print(f"### {harness['name']}")
    reason = blocker(harness)
    if reason:
        print(f"    NOT RUN here ({reason}).")
        print(f"    To run it, provide: {harness['needs']}")
        print()
        continue
    previous_adapter_python = os.environ.get("ADAPTER_PYTHON")
    try:
        adapter_python = harness.get("python")
        if adapter_python:
            os.environ["ADAPTER_PYTHON"] = str(adapter_python)
        else:
            os.environ.pop("ADAPTER_PYTHON", None)
        cfg = for_harness(harness)
        plan = fabric.plan(cfg, base_dir=BASE_DIR)
        rc = plan.config.runtime
        model = plan.config.models["default"]["model"]
        print(f"    adapter = {plan.adapter.adapter_id}")
        print(f"    model   = {model}   input schema = {rc.input_schema}")
        try:
            result = await fabric.run(cfg, base_dir=BASE_DIR, input=PROMPT)
            print(f"    RAN -> {result.status}")
            print(f"    reply: {oneline(getattr(result.output, 'response', result.output))}")
            if result.status != "succeeded":
                detail = failure_detail(result)
                print("    error:", json.dumps(detail, indent=2))
                run_failures.append(f"{harness['name']}: {json.dumps(detail, sort_keys=True)}")
        except Exception as error:
            print(f"    run failed ({type(error).__name__}: {oneline(error, 120)}).")
            print("    Attempted run failed; continuing with remaining variants.")
            run_failures.append(f"{harness['name']}: {type(error).__name__}: {error}")
        print()
    except Exception as error:
        print(f"    planning failed ({type(error).__name__}: {oneline(error, 120)}).")
        print("    Attempted planning failed; continuing with remaining variants.")
        run_failures.append(f"{harness['name']}: {type(error).__name__}: {error}")
        print()
    finally:
        if previous_adapter_python is None:
            os.environ.pop("ADAPTER_PYTHON", None)
        else:
            os.environ["ADAPTER_PYTHON"] = previous_adapter_python

## Group 2 — Agent configuration variation

The other kind of variation keeps the harness fixed and composes on the
config. Starting from the maintained `base_config()` (which carries a
code-review skill), you strip or add skills and an optional MCP server and swap
the model — all on a **copy**, leaving the base untouched.

In [ ]:
def summarize(cfg):
    return {
        "model": cfg.models["default"].model,
        "skills": [Path(p).name for p in (cfg.skills.paths if cfg.skills else [])],
        "mcp_servers": list(cfg.mcp.servers) if cfg.mcp else [],
    }

base = base_config()
print("as shipped :", summarize(base))

# Strip capabilities on a copy.
stripped = base.model_copy(deep=True)
stripped.remove_skill_path("./skills/code-review")
print("stripped   :", summarize(stripped))

# Compose on another copy: add the real example skill, optionally add the
# configured GitHub MCP server, and swap the model.
recomposed = stripped.model_copy(deep=True)
recomposed.add_skill_path("./skills/code-review")
if github_mcp_url := os.environ.get("GITHUB_MCP_URL"):
    recomposed.add_mcp_server(
        "github", transport="streamable-http",
        url=github_mcp_url, exposure="harness_native",
    )
recomposed.models["default"] = ModelConfig(
    provider="nvidia", model="nvidia/nemotron-3-super-49b-a5b", api_key_env="NVIDIA_API_KEY"
)
print("recomposed :", summarize(recomposed))

print("base intact:", summarize(base))

### Telemetry is a configuration variation too

Turning on NeMo Relay tracing is the same pattern: clone the agent and set a
field. `enable_relay(...)` is a public `FabricConfig` method — no example
helper involved — so you can see exactly what telemetry adds to the config.
When the Relay dependency is present in the adapter environment, the run
writes trace files under the notebook's artifact root, which we print in full.

We use the Deep Agents agent here because its adapter environment has
`nemo_relay` installed.

In [ ]:
relay_dir = REPO_ROOT / "examples" / "notebooks" / "artifacts" / "relay"
deep = HARNESSES[1]  # Deep Agents
reason = blocker(deep)

if reason or not RELAY_AVAILABLE:
    missing = reason or "nemo_relay is not installed in the adapter environment"
    print(f"Not emitting telemetry here ({missing}).")
    print("Install the Relay extra (nemo-fabric[relay]) in the adapter env to")
    print("produce ATOF trace files under", relay_dir)
else:
    previous_adapter_python = os.environ.get("ADAPTER_PYTHON")
    try:
        os.environ["ADAPTER_PYTHON"] = str(deep["python"])
        if relay_dir.exists():
            shutil.rmtree(relay_dir)
        # Telemetry is just another field: clone the agent and enable Relay.
        traced = for_harness(deep)
        traced.enable_relay(
            output_dir=str(relay_dir),
            observability=RelayObservabilityConfig(
                atof=RelayAtofConfig(
                    enabled=True,
                    sinks=[RelayAtofFileSinkConfig(
                        output_directory=str(relay_dir),
                        filename="events.atof.jsonl", mode="overwrite",
                    )],
                ),
            ),
        )
        result = await fabric.run(traced, base_dir=BASE_DIR, input=PROMPT)
        if result.status != "succeeded":
            detail = failure_detail(result)
            raise RuntimeError(f"Deep Agents Relay run failed: {json.dumps(detail, sort_keys=True)}")
        print("run status     :", result.status)
        telemetry_providers = [t.provider for t in result.telemetry]
        if "relay" not in telemetry_providers:
            raise RuntimeError("Deep Agents Relay run returned no Relay telemetry reference")
        print("telemetry refs :", telemetry_providers)
        traces = sorted(p for p in relay_dir.rglob("*") if p.is_file())
        atof_paths = sorted(relay_dir.rglob("events.atof.jsonl"))
        if not atof_paths:
            raise RuntimeError("Deep Agents Relay run produced no ATOF trace")
        for atof_path in atof_paths:
            atof_lines = [line for line in atof_path.read_text().splitlines() if line.strip()]
            if not atof_lines:
                raise RuntimeError(f"Deep Agents Relay run produced an empty ATOF trace: {atof_path}")
            for line in atof_lines:
                json.loads(line)
        print("trace files    :", len(traces))
        for path in traces:
            print("   ", path.relative_to(REPO_ROOT))

        # Print the full Relay trace: every ATOF event.
        for path in atof_paths:
            print(f"\n===== {path.relative_to(REPO_ROOT)} =====")
            lines = [line for line in path.read_text().splitlines() if line.strip()]
            for i, line in enumerate(lines, 1):
                print(f"\n--- event {i}/{len(lines)} ---")
                print(json.dumps(json.loads(line), indent=2))
    except Exception as error:
        print(f"Deep Agents Relay failed ({type(error).__name__}: {oneline(error, 120)}).")
        run_failures.append(f"Deep Agents Relay: {type(error).__name__}: {error}")
    finally:
        if previous_adapter_python is None:
            os.environ.pop("ADAPTER_PYTHON", None)
        else:
            os.environ["ADAPTER_PYTHON"] = previous_adapter_python

if run_failures:
    raise RuntimeError("Notebook execution failures:\n- " + "\n- ".join(run_failures))

## Recap

Building on the maintained [code-review example](../code_review_agent/README.md),
you saw two kinds of variation on one typed config:

- **Harness variation** — the same agent ran on different harnesses and
  returned the same `RunResult` shape, so your application code does not
  change when you compare or switch harnesses.
- **Agent configuration variation** — skills, MCP servers, models, and
  telemetry are all fields you compose onto a copy of the config, which is
  the basis for evaluation and ablation sweeps.

Throughout, NeMo Fabric's runtime contract stayed the same — only the harness or
the configuration changed. For the full API, see the
[Python SDK guide](../../docs/sdk/python.mdx).